# Budget Analysis V.3

# Package Import

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

import emoji
import re

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from datetime import date, time, timedelta, datetime
from dateutil.relativedelta import relativedelta

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Database Connection

In [2]:
try:
    conn = create_engine('sqlite:///money_android.sqlite')
    try:
        test_con = '''select name as tables from sqlite_master where type="table"'''
        display(pd.read_sql(test_con, conn))
        del test_con
    except Exception as e:
        print(f'Error Retrieving data: {e}')
except Exception as e:
    print(f'Database Connection fail: {e}')

,tables
0,android_metadata
1,MESSAGEMACRO2
2,sqlite_sequence
3,INOUTCOME
4,ASSETS
5,ZCATEGORY
6,ZETC
7,SMS_RAW_READ
8,REPEATTRANSACTION
9,FAVTRANSACTION


# Data Import

In [3]:
bdgt_tbl = pd.read_sql('select * from INOUTCOME', conn)
cat_tbl = pd.read_sql('select * from ZCATEGORY', conn)
asset_tbl = pd.read_sql('select * from ASSETS', conn)

display(bdgt_tbl, cat_tbl, asset_tbl)

,AID,uid,assetUid,CARDDIVIDMONTH,ctgUid,toAssetUid,ZCONTENT,ZDATE,WDATE,wtime,paid,DO_TYPE,ZMONEY,txUidTrans,ZDATA,SMS_RDATE,IN_ZMONEY,ASSET_NIC,CATEGORY_NAME,cardDivideUid,CARD_DIVIDE_MONTH_STR,MARK,txUidFee,SMS_ORIGIN,SMS_PARSE_CONTENT,IS_DEL,UTIME,currencyUid,AMOUNT_ACCOUNT,lat,lng,gstd,isSynced,syncTime,syncVersion
0,1,c8638ef1-5755-4492-87e0-fc2cc234791e,11,0,9,,,1726617607841,2024-09-18,None,None,1,1475.23,,,,25.0,None,None,,,0,None,None,None,0,1736642830928,PHP_USD,25.00,None,None,None,0,None,0
1,2,845207f5-c8ae-4116-8e62-20ece9ca2c2f,11,0,6,,Don Lemon 🍋,1726632007841,2024-09-18,None,None,1,1711.26,,,,29.0,None,None,,,0,None,None,None,0,1736642830928,PHP_USD,29.00,None,None,None,0,None,0
2,3,011c5f04-6c3e-4bce-9e63-e303c4ac570a,2,0,6,,Samgyup,1726380441057,2024-09-15,None,None,1,116660.8,,,,1977.0,None,None,,,0,None,None,None,0,1736642830928,PHP_USD,1977.00,None,None,None,0,None,0
3,4,1d502cea-0868-4243-a6e2-c4ec06dc0717,11,0,6,,Samgyup,1726227055993,2024-09-13,None,None,1,35346.39,,,,599.0,None,None,,,0,None,None,None,0,1736642830928,PHP_USD,599.00,None,None,None,0,None,0
4,5,88eb9171-fd63-4c0e-95b9-c5d149e2e01c,1,0,17,,Udemy + Corn Kape,1726231445059,2024-09-13,None,None,1,76711.71,,,,1300.0,None,None,,,0,None,None,None,0,1736642830928,PHP_USD,1300.00,None,None,None,0,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,569,517669f2-2a38-425e-affd-0efefe3dba8d,11,0,8087c509-180d-4b4a-ac10-468a0f868662,,Lunch,1744072365017,2025-04-08,None,None,1,238.0,,,,238.0,None,None,,,0,None,None,None,0,1744072621865,PHP_PHP,4.03,None,None,None,0,None,0
564,571,d15c5bd1-c8bd-4587-835a-1f1f68da6269,1,0,,11,Withdraw,1744072692072,2025-04-08,None,None,3,200.0,0165529b-7676-454b-8440-c194efaae437,,,200.0,None,None,,,0,12943b9e-9e78-476e-8d4c-1c84ff2a349e,None,None,0,1744072705416,PHP_PHP,3.39,None,None,None,0,None,0
565,572,f025f239-8520-4737-a8a2-3b9dbb854331,11,0,,1,Withdraw,1744072692072,2025-04-08,None,None,4,200.0,0165529b-7676-454b-8440-c194efaae437,,,200.0,None,None,,,0,12943b9e-9e78-476e-8d4c-1c84ff2a349e,None,None,0,1744072705420,PHP_PHP,3.39,None,None,None,0,None,0
566,573,1c540e4f-e1fa-4809-92f0-18854c75c308,11,0,e23aa8f6-06d9-4b33-a1c5-e696faa16224,,Snack (Office),1744114014015,2025-04-08,None,None,1,50.0,,,,50.0,None,None,,,0,None,None,None,0,1744114032327,PHP_PHP,0.85,None,None,None,0,None,0


,ID,C_IS_DEL,C_UTIME,uid,NAME,ORDERSEQ,TYPE,STATUS,pUid,isSynced,syncTime,syncVersion
0,1,None,1726638920691,1,🤑 Allowance,1,0,0,0,0,None,0
1,2,None,1726638920691,2,💰 Salary,2,0,0,0,0,None,0
2,3,None,1726638920693,3,💵 Petty cash,3,0,0,0,0,None,0
3,4,None,1726638920694,4,🏅 Bonus,4,0,0,0,0,None,0
4,5,None,1726638920695,5,Other,5,0,0,0,0,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...
64,65,None,1739582285466,e23aa8f6-06d9-4b33-a1c5-e696faa16224,Snacks 🍟,5,1,2,6,0,None,0
65,66,None,1739106862285,7b0c7dac-639f-41b4-a7f5-4e34a654a1fc,Internet 💻,66,1,2,4076de27-f668-41e8-ac5c-dcc0c69b7cae,0,None,0
66,67,None,1739582090198,42d4ac1f-866b-4e15-820b-6a7b762ec602,Incentives,67,0,0,,0,None,0
67,68,None,1739582285460,8087c509-180d-4b4a-ac10-468a0f868662,Breakfast,0,1,2,6,0,None,0


,ID,CARD_DAY_FIN,CARD_DAY_PAY,NIC_NAME,ORDERSEQ,ZDATA,ZDATA1,ZDATA2,IS_TRANS_EXPENSE,IS_CARD_AUTO_PAY,APP_PACKAGE,APP_NAME,SMS_TEL,SMS_STRING,A_UTIME,CARD_USAGE_HURDLE_TYPE,CARD_USAGE_HURDLE_START_DATE,CARD_USAGE_HURDLE_AMOUNT,uid,currencyUid,cardAssetUid,groupUid,isSynced,syncTime,syncVersion
0,1,1,1,Cash,1,,,0,0,0,None,None,,,1736642830921,1,None,0.0,11,PHP_USD,,11,0,None,None
1,2,1,1,Accounts,2,,,0,0,0,None,None,,,1736642830924,1,None,0.0,1,PHP_USD,,1,0,None,None
2,3,1,1,Card,3,,,0,0,0,None,None,,,1736642830925,1,None,0.0,2,PHP_USD,1,2,0,None,None


# Data Normalization

## Data Mapping

In [6]:
cat_dict = dict(zip(cat_tbl['uid'], cat_tbl['NAME']))
asset_dict = dict(zip(asset_tbl['uid'],asset_tbl['NIC_NAME']))
do_dict = {'0':'Income', '1':'Expense', '3':'Transfer From', '4':'Transfer To'}

bdgt_tbl['Category'] = bdgt_tbl['ctgUid'].map(cat_dict)
bdgt_tbl['Assets'] = bdgt_tbl['assetUid'].map(asset_dict)
bdgt_tbl['Fin_Transaction'] = bdgt_tbl['DO_TYPE'].map(do_dict)

final_bdg = bdgt_tbl[['uid', 'Fin_Transaction', 'Assets', 'Category', 'ZCONTENT', 'WDATE', 'IN_ZMONEY']]
final_bdg.rename(columns= {'ZCONTENT':'Notes', 'WDATE':'Dates', 'IN_ZMONEY':'Amount'}, inplace=True)

In [9]:
final_bdg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568 entries, 0 to 567
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   uid              568 non-null    object
 1   Fin_Transaction  568 non-null    object
 2   Assets           568 non-null    object
 3   Category         472 non-null    object
 4   Notes            568 non-null    object
 5   Dates            568 non-null    object
 6   Amount           568 non-null    object
dtypes: object(7)
memory usage: 31.2+ KB


In [8]:
final_bdg

,uid,Fin_Transaction,Assets,Category,Notes,Dates,Amount
0,c8638ef1-5755-4492-87e0-fc2cc234791e,Expense,Cash,🚖 Transport,,2024-09-18,25.0
1,845207f5-c8ae-4116-8e62-20ece9ca2c2f,Expense,Cash,🍜 Food,Don Lemon 🍋,2024-09-18,29.0
2,011c5f04-6c3e-4bce-9e63-e303c4ac570a,Expense,Card,🍜 Food,Samgyup,2024-09-15,1977.0
3,1d502cea-0868-4243-a6e2-c4ec06dc0717,Expense,Cash,🍜 Food,Samgyup,2024-09-13,599.0
4,88eb9171-fd63-4c0e-95b9-c5d149e2e01c,Expense,Accounts,Other,Udemy + Corn Kape,2024-09-13,1300.0
...,...,...,...,...,...,...,...
563,517669f2-2a38-425e-affd-0efefe3dba8d,Expense,Cash,Breakfast,Lunch,2025-04-08,238.0
564,d15c5bd1-c8bd-4587-835a-1f1f68da6269,Transfer From,Accounts,NaN,Withdraw,2025-04-08,200.0
565,f025f239-8520-4737-a8a2-3b9dbb854331,Transfer To,Cash,NaN,Withdraw,2025-04-08,200.0
566,1c540e4f-e1fa-4809-92f0-18854c75c308,Expense,Cash,Snacks 🍟,Snack (Office),2025-04-08,50.0


## Data Normalization / Standardization

In [14]:
def convert_dtypes(data, col1, col2):
    data[col1] = pd.to_datetime(data[col1], format="%Y-%m-%d", errors='coerce')
    data[col2] = data[col2].astpye('float64')

def preprocess(data, col):
    """
    Converting empty spaces to None Values    
    """
    data = data.applymap(lambda x: x.strip() if isinstance(x, str) and x.strip() != '' else (None if isinstance(x, str) else x))
    def remove_emojis(textdata):
        """
        Removal of emojis
        """
        if isinstance(textdata, str):
            if textdata is not None:
                return emoji.replace_emojis(textdata, replace='').strip()
        return textdata
    data[col] = data[col].apply(remove_emojis)
    return data